# a2_write

> Phase 21 (`specs/multiagent/phase21/`): A2's report and figure from `a2Summaries` (kept apart so `a2_report` stays under 200 lines, as Phase 20's `bridge_write`). `writeA2` writes `report.md`: Gate A2 with the manipulation check and H21-2 beside it, every arm's contrast, persistence, division of labour, volunteers and specialization (H21-4), the core measures, P2 (H21-3 with Holm), what is not measured, the claim rules and every other `A2_PLAN` item; `figA2` draws `a2.png` (guarding of the success and failure guards by board and arm; P2's suppression by size and prompt).

In [ ]:
#| default_exp a2_write

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json

from explRL.a2_grid import A2_PLAN, P2_N


def _g(x, d=3):
    try:
        return f'{float(x):.{d}f}'
    except (TypeError, ValueError):
        return str(x)


def _ci(v):
    return f"{_g(v['estimate'])} [{_g(v['lo'])}, {_g(v['hi'])}] (n {v['n']})"


CORE = ('gullibility', 'suppression', 'contestsPerLie', 'timeToContest', 'guardedLies', 'errorCorr', 'collaboration', 'welfare',
        'welfareHonest', 'welfarePerKTok', 'tokensPerSolved', 'boards')


def writeA2(path, S, meta, plan=A2_PLAN):
    "report.md: Gate A2, H21-2 to H21-4, the manipulation check, the core measures, P2, what is not measured and the plan's items."
    G, M = S['gate'], S['manipulation']
    L = ['# Phase 21 (A2): experience and rule-text screens (exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}, cost ${_g(meta.get('cost', 0), 2)}", '',
         '## Gate A2 (H21-1)', '',
         f"Manipulation check: {M['passed']} of {M['of']} teams had both success guards contest on boards 1-2 with a correct answer "
         f"following (needs {plan['manipulation']['teams']}): {'read' if M['read'] else 'NOT READ'}."]
    if 'H21_1' in G:
        L += [f"H21-1, own minus shared contrast in guarding on boards 3-6 (one-sided exact sign-flip over {len(G['H21_1']['perTeam'])} teams): "
              f"{_ci(G['H21_1']['est'])}, p {_g(G['H21_1']['p'], 4)}. **Gate A2 {'passes' if G['passes'] else 'fails'}.**",
              f"H21-2, own-record contrast above 0 (reported beside the gate, P21-5): p {_g(G['H21_2']['p'], 4)}, "
              f"{'supported' if G['H21_2']['supported'] else 'not supported'}."]
    L += [f"Note: {G['note']}" if G.get('note') else '', '', '## Arms on boards 3-6 (H21-4)', '',
          '| arm | contrast (success - failure guarding) | p vs 0 | persistence (5 pairs) | persistence (boards 3-6) | division of labour | volunteers | specialization - null |',
          '|---|---|---|---|---|---|---|---|']
    for a, v in S.get('arms', {}).items():
        L.append(f"| {a} | {_ci(v['contrast'])} | {_g(v['pVs0'], 4)} | {_ci(v['persistence'])} | {_ci(v['persistence36'])} | {_ci(v['dol'])} | "
                 f"{_g(v['volunteers'])} | {_g(v['specExcess'])} |")
    L += ['', '## Core measures (B1)', '', '| set | ' + ' | '.join(CORE) + ' |', '|---|' + '---|' * len(CORE)]
    sets = [(f'{a}, boards 3-6', v['core']) for a, v in S.get('arms', {}).items()] + [('boards 1-2 (shared by the arms)', S['boards12'])]
    sets += [(f'P2 N {N} {m}', S['p2'][N]['core'][m]) for N in P2_N for m in ('orig', 'drop')]
    L += [f"| {n} | " + ' | '.join(_g(c.get(k)) for k in CORE) + ' |' for n, c in sets]
    L += ['', f"Lies on a failure guard's clue on boards 1-2: {S['boards12']['liarOnFailure']} (must be 0).", '',
          '## Rule text (P2, H21-3; Holm across N 8 and 32)', '',
          '| N | drop - orig suppressedHeld | p | p (Holm) | supported | suppressedHeld orig / drop | suppressed (Phase 15) orig / drop | first wrong per honest turn orig / drop |',
          '|---|---|---|---|---|---|---|---|']
    for N in P2_N:
        v = S['p2'][N]
        L.append(f"| {N} | {_ci(v['diff'])} | {_g(v['p'], 4)} | {_g(v['pHolm'], 4)} | {v['supported']} | "
                 + ' | '.join(f"{_g(v[k]['orig'])} / {_g(v[k]['drop'])}" for k in ('suppressedHeld', 'suppressed', 'firstWrongPerTurn')) + ' |')
    L += ['', f"Pooled over both sizes: {_g(S['p2Pooled']['diff'])} ({S['p2Pooled']['n']} pairs).", '',
          '## Not measured', '', f"{', '.join(S['notMeasured'])}: no A2 board is clean (P21-6 chose liar boards only, P1's boards "
          'always have liars) and A2 has no squads. These are recorded gaps.', '', '## Claim rules', ''] + [f'- {c}' for c in plan['claims']]
    L += ['', '## Plan items', ''] + [f"- **{k}**: {json.dumps(v, default=str)}" for k, v in plan.items()]
    open(path, 'w').write('\n'.join(L) + '\n')


def figA2(out, S):
    "a2.png: guarding of the success (solid) and failure (dashed) guards by board and arm; P2's suppressedHeld by size and prompt."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw=dict(width_ratios=[3, 2]))
    col = dict(none='#7f7f7f', shared='#1f6391', own='#c0392b')
    for arm, v in S.get('arms', {}).items():
        bs = sorted(v['guarding'])
        a.plot(bs, [v['guarding'][x][0] for x in bs], '-o', color=col.get(arm), ms=4, label=f'{arm}: success guards')
        a.plot(bs, [v['guarding'][x][1] for x in bs], '--', color=col.get(arm), label=f'{arm}: failure guards')
    a.axvline(2.5, color='#555555', lw=0.8, ls=':')
    a.set(xlabel='board (records from board 3)', ylabel='guarding (held-clue posts / turns)',
          title=f"Gate A2 {'passes' if S['gate']['passes'] else 'fails' if S['gate']['read'] else 'not read'}")
    a.legend(fontsize=6, frameon=False, ncol=2)
    for i, N in enumerate(P2_N):
        v = S['p2'][N]['suppressedHeld']
        b.bar([3 * i, 3 * i + 1], [v['orig'], v['drop']], 0.8, color=['#7f7f7f', '#2e8b57'])
    b.set_xticks([0, 1, 3, 4], ['N 8 orig', 'N 8 drop', 'N 32 orig', 'N 32 drop'], fontsize=8)
    b.set(ylabel='liar-first posts left uncorrected', title='Re-sharing sentence (H21-3)', ylim=(0, 1))
    for ax in (a, b):
        ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(f'{out}/a2.png', dpi=150, metadata={'Software': None})
    plt.close(fig)

## Tests

In [ ]:
# test_a2_report (AC-9, FR-C3, FR-C4): on syntheticRows21 the sign-flip test, Holm and the contrast recover known effects;
# the report lists every A2_PLAN item, the core measures and the not-measured note; a null effect fails the gate; a failed
# manipulation check leaves the gate unread; partial='stopped' reports P2 and 'Gate A2 not read'; missing boards refuse
import tempfile
from pathlib import Path
from explRL.a2_report import a2Summaries, a2TeamEstimates
from explRL.a2_synth import pilotSigma, power21, syntheticRows21
from explRL.a2_write import *

rows, p2 = syntheticRows21()
S = a2Summaries(rows, p2, nboot=200)
assert S['gate']['passes'] and S['gate']['H21_1']['p'] == 1 / 256 and S['gate']['H21_2']['supported'] and S['manipulation']['passed'] == 8
assert abs(S['arms']['own']['contrast']['estimate'] - 0.2) < 0.03 and abs(S['arms']['shared']['contrast']['estimate']) < 0.03
assert all(S['p2'][N]['supported'] and S['p2'][N]['pHolm'] == 2 / 256 and abs(S['p2'][N]['diff']['estimate'] + 0.3) < 0.05 for N in (8, 32))
assert set(S['arms']['own']['core']) >= {'gullibility', 'suppression', 'contestsPerLie', 'timeToContest', 'errorCorr', 'welfarePerKTok', 'tokensPerSolved'}
S0 = a2Summaries(*syntheticRows21(dict(own=0.0, p2=0.0)), nboot=200)
assert not S0['gate']['passes'] and S0['gate']['H21_1']['p'] > 0.05 and not any(S0['p2'][N]['supported'] for N in (8, 32))
Sm = a2Summaries(*syntheticRows21(manipTeams=5), nboot=200)
assert not Sm['gate']['read'] and not Sm['gate']['passes'] and 'not read' in Sm['gate']['note']
stop = [r for r in rows if int(r['pos']) < 2]
Ss = a2Summaries(stop, p2, partial='stopped', nboot=200)
assert not Ss['gate']['read'] and 'not read' in Ss['gate']['note'] and Ss['p2'][8]['supported'] and 'arms' not in Ss
try:
    a2Summaries(stop, p2, nboot=200)
    raise AssertionError('missing boards must refuse')
except ValueError as e:
    assert 'missing boards' in str(e)
E = a2TeamEstimates([r for r in rows if int(r['seqId']) == 2100 and r['arm'] == 'own'])   # pilot-sized rows: no coverage check
assert set(E['contrast']) == {'own'} and abs(E['contrast']['own'][2100] - 0.2) < 0.05 and E['gate'] == {}
with tempfile.TemporaryDirectory() as d:
    for s, mode in ((S, 'quick'), (Ss, 'stopped')):
        writeA2(Path(d) / 'report.md', s, dict(mode=mode, planHash='x', commit='y', cost=0.0))
        figA2(d, s)
        txt = (Path(d) / 'report.md').read_text()
        assert all(f'**{k}**' in txt for k in A2_PLAN) and 'cost without liars' in txt and 'damage won back' in txt and 'Gate A2' in txt
        assert (Path(d) / 'a2.png').stat().st_size > 0
    assert 'Gate A2 not read: the run stopped after wave 2' in txt
P = power21(pilotSigma([r for r in rows if int(r['seqId']) == 2100 and r['arm'] == 'own']), nsim=200)
assert P['power']['gate'][0.3] > 0.9 and P['power']['gate'][0.05] <= P['power']['gate'][0.3]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()